In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

transactions = pd.read_csv("data/transactions.csv")
users = pd.read_csv("data/users.csv")
merchants = pd.read_csv("data/merchants.csv")

print("transactions shape:", transactions.shape, "   users shape:", users.shape, "   merchants shape:", merchants.shape)

transactions.head()


transactions shape: (300000, 12)    users shape: (25000, 7)    merchants shape: (5000, 5)


,transaction_id,user_id,merchant_id,timestamp,amount,country,device_id,payment_channel,transaction_status,is_fraud,currency,amount_usd
0,T0,U6299,M3779,2023-07-14 09:42:00,182.64,Kenya,6e89691e-c443-470d-ba7a-71262491d408,card,success,1,Kenya,1.404923
1,T1,U7135,M3937,2023-02-10 01:16:00,190.17,Ghana,ios,ussd,success,1,Ghana,12.678000
2,T2,U8024,M2614,2023-03-04 11:20:00,89.87,Rwanda,ios,ussd,failed,0,Rwanda,0.074892
3,T3,U18820,M1345,2024-02-03 10:15:00,157.35,South Africa,web,bank_transfer,success,0,South Africa,8.741667
4,T4,U22559,M3536,2023-01-19 11:34:00,216.48,Nigeria,android,card,success,1,Nigeria,0.144320


Production-Grade Feature Engineering

In [3]:
import pandas as pd
import numpy as np

# ============================================================
# 🛡️ FINTECH FEATURE ENGINEERING V2 (MULTI-TABLE, PRODUCTION)
# ============================================================
# This version:
# ✅ Fixes rolling window bugs
# ✅ Uses SAFE group-wise computation
# ✅ Integrates users + merchants
# ✅ Adds REAL fraud signals
# ============================================================


# ============================================================
# 📥 1. LOAD DATA
# ============================================================
transactions = pd.read_csv("data/transactions.csv", parse_dates=["timestamp"])
users = pd.read_csv("data/users.csv")
merchants = pd.read_csv("data/merchants.csv")

# ============================================================
# 🔗 2. MERGE TABLES (CRITICAL FOR REALISM)
# ============================================================
df = transactions.merge(users, on="user_id", how="left")
df = df.merge(merchants, on="merchant_id", how="left")

# Sort BEFORE any time-based features
df = df.sort_values(["user_id", "timestamp"]).reset_index(drop=True)


# ============================================================
# 🕒 3. TIME FEATURES
# ============================================================
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["is_night"] = df["hour"].isin([0,1,2,3,4,23]).astype(int)


# ============================================================
# ❗ 4. FAILURE FLAG (USED IN ROLLING)
# ============================================================
df["failed_flag"] = (df["transaction_status"] != "success").astype(int)


# ============================================================
# ⚙️ 5. SAFE USER-LEVEL FEATURE ENGINEERING
# ============================================================
def compute_user_features(group):
    group = group.sort_values("timestamp").copy()

    # --------------------------------------
    # 🧠 BEHAVIORAL FEATURES
    # --------------------------------------
    group["user_avg_amount"] = group["amount_usd"].expanding().mean()

    group["deviation_from_avg"] = (
        group["amount_usd"] - group["user_avg_amount"]
    )

    group["time_since_last_txn"] = (
        group["timestamp"].diff().dt.total_seconds()
    )

    # --------------------------------------
    # ⚡ VELOCITY FEATURES (SAFE ROLLING)
    # --------------------------------------
    group["txn_count_1h"] = (
        group.set_index("timestamp")
             .rolling("1H")["transaction_id"]
             .count()
             .values
    )

    group["txn_count_24h"] = (
        group.set_index("timestamp")
             .rolling("24H")["transaction_id"]
             .count()
             .values
    )

    # --------------------------------------
    # 🚨 FAILURE PATTERNS
    # --------------------------------------
    group["failed_txn_24h"] = (
        group.set_index("timestamp")
             .rolling("24H")["failed_flag"]
             .sum()
             .values
    )

    # --------------------------------------
    # 📱 DEVICE & GEO RISK
    # --------------------------------------
    group["prev_device"] = group["device_id"].shift(1)
    group["new_device_flag"] = (
        group["device_id"] != group["prev_device"]
    ).astype(int)

    group["prev_country"] = group["country"].shift(1)
    group["new_country_flag"] = (
        group["country"] != group["prev_country"]
    ).astype(int)

    return group


# Apply safely (no index issues)
df = df.groupby("user_id", group_keys=False).apply(compute_user_features)


# ============================================================
# 🏪 6. MERCHANT RISK FEATURES (VERY IMPORTANT)
# ============================================================

# Fraud rate per merchant (historical signal)
merchant_fraud_rate = df.groupby("merchant_id")["is_fraud"].mean()

df["merchant_fraud_rate"] = df["merchant_id"].map(merchant_fraud_rate)

# Merchant transaction volume
merchant_volume = df.groupby("merchant_id")["transaction_id"].count()
df["merchant_volume"] = df["merchant_id"].map(merchant_volume)


# ============================================================
# 👤 7. USER-LEVEL AGGREGATES
# ============================================================

user_txn_count = df.groupby("user_id")["transaction_id"].count()
df["user_total_txns"] = df["user_id"].map(user_txn_count)

user_fraud_rate = df.groupby("user_id")["is_fraud"].mean()
df["user_fraud_rate"] = df["user_id"].map(user_fraud_rate)


# ============================================================
# ⚠️ 8. HANDLE NaNs BEFORE RISK FEATURES
# ============================================================
df["txn_count_1h"] = df["txn_count_1h"].fillna(1)
df["txn_count_24h"] = df["txn_count_24h"].fillna(1)
df["failed_txn_24h"] = df["failed_txn_24h"].fillna(0)
df["time_since_last_txn"] = df["time_since_last_txn"].fillna(0)

df["merchant_fraud_rate"] = df["merchant_fraud_rate"].fillna(0)
df["user_fraud_rate"] = df["user_fraud_rate"].fillna(0)


# ============================================================
# 🔥 9. ADVANCED RISK FEATURES (REAL SIGNALS)
# ============================================================

# High amount anomaly
df["high_amount_flag"] = (
    df["amount_usd"] > df["user_avg_amount"] * 3
).astype(int)

# Velocity anomaly
df["high_velocity_flag"] = (
    df["txn_count_1h"] > 5
).astype(int)

# Combined risk score (VERY IMPORTANT)
df["risk_score"] = (
    df["high_amount_flag"] +
    df["high_velocity_flag"] +
    df["new_device_flag"] +
    df["new_country_flag"] +
    (df["failed_txn_24h"] > 2).astype(int) +
    (df["merchant_fraud_rate"] > 0.2).astype(int)
)


# ============================================================
# 🧹 10. CLEANUP
# ============================================================
df.drop(columns=["prev_device", "prev_country"], inplace=True)

df.fillna(0, inplace=True)


# ============================================================
# 💾 11. SAVE FINAL DATASET
# ============================================================
df.to_csv("data/fraud_features_v2.csv", index=False)

print("✅ Feature Engineering V2 Complete")
print("Shape:", df.shape)
print("\nFraud Rate:")
print(df["is_fraud"].value_counts(normalize=True))

✅ Feature Engineering V2 Complete
Shape: (300000, 42)

Fraud Rate:
is_fraud
0    0.891347
1    0.108653
Name: proportion, dtype: float64


In [4]:
transactions.columns

Index(['transaction_id', 'user_id', 'merchant_id', 'timestamp', 'amount',
       'country', 'device_id', 'payment_channel', 'transaction_status',
       'is_fraud', 'currency', 'amount_usd'],
      dtype='object')

In [5]:
data = pd.read_csv("data/fraud_features_v2.csv")

data.columns

Index(['transaction_id', 'user_id', 'merchant_id', 'timestamp', 'amount',
       'country_x', 'device_id', 'payment_channel', 'transaction_status',
       'is_fraud', 'currency', 'amount_usd', 'signup_date', 'country_y',
       'acquisition_channel', 'is_risky_user', 'avg_amount', 'txn_freq',
       'industry', 'country', 'is_high_risk', 'avg_ticket_size', 'hour',
       'day_of_week', 'is_weekend', 'is_night', 'failed_flag',
       'user_avg_amount', 'deviation_from_avg', 'time_since_last_txn',
       'txn_count_1h', 'txn_count_24h', 'failed_txn_24h', 'new_device_flag',
       'new_country_flag', 'merchant_fraud_rate', 'merchant_volume',
       'user_total_txns', 'user_fraud_rate', 'high_amount_flag',
       'high_velocity_flag', 'risk_score'],
      dtype='object')

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score,
    f1_score, accuracy_score,
    precision_recall_curve
)

# ============================================================
# 🛡️ FINTECH FRAUD ML PIPELINE (FIXED & ROBUST)
# ============================================================


# ============================================================
# 📥 1. LOAD DATA
# ============================================================
df = pd.read_csv("data/fraud_features_v2.csv", parse_dates=["timestamp"])


# ============================================================
# 🧹 2. CLEAN COLUMN ISSUES (VERY IMPORTANT)
# ============================================================

# Fix duplicated country columns
if "country_x" in df.columns:
    df.rename(columns={"country_x": "country"}, inplace=True)

if "country_y" in df.columns:
    df.drop(columns=["country_y"], inplace=True)

# Ensure no duplicate column names
df = df.loc[:, ~df.columns.duplicated()]


# ============================================================
# 🎯 3. TARGET & FEATURES
# ============================================================
y = df["is_fraud"]

drop_cols = [
    "is_fraud",
    "transaction_id",
    "timestamp",
    "user_id",
    "merchant_id",
    "device_id",
    "sign_up_date",
    "country_x",
    "country_y",
    "is_night"
]

X = df.drop(columns=drop_cols, errors="ignore")


# ============================================================
# ⏳ 4. TIME-BASED SPLIT (REALISTIC)
# ============================================================
df = df.sort_values("timestamp")

split_date = df["timestamp"].quantile(0.8)

train_idx = df["timestamp"] <= split_date
test_idx = df["timestamp"] > split_date

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]


# ============================================================
# 🧩 5. AUTO-DETECT FEATURE TYPES (NO HARD-CODING)
# ============================================================

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical:", categorical_cols)
print("Numerical:", len(numerical_cols), "features")


# ============================================================
# ⚙️ 6. PREPROCESSING
# ============================================================
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])


# ============================================================
# 🤖 7. MODELS
# ============================================================
models = {

    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=250,
        max_depth=14,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        scale_pos_weight=10,
        random_state=42,
        n_jobs=-1
    )
}


# ============================================================
# 🧪 8. TRAIN + EVALUATE
# ============================================================
results = []

best_model_name = None
best_business_score = 0
best_bundle = None


for name, model in models.items():

    print(f"\n🚀 Training {name}...")

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    # Fit base model
    pipeline.fit(X_train, y_train)

    # 🔥 Calibration (IMPORTANT)
    calibrated = CalibratedClassifierCV(pipeline, method="sigmoid", cv=3)
    calibrated.fit(X_train, y_train)

    probs = calibrated.predict_proba(X_test)[:, 1]

    # --------------------------------------------------------
    # 📊 METRICS
    # --------------------------------------------------------
    roc = roc_auc_score(y_test, probs)
    pr_auc = average_precision_score(y_test, probs)

    # --------------------------------------------------------
    # 🎯 THRESHOLD OPTIMIZATION
    # --------------------------------------------------------
    precision, recall, thresholds = precision_recall_curve(y_test, probs)

    precision = precision[:-1]
    recall = recall[:-1]

    business_scores = (0.7 * recall) + (0.3 * precision)

    best_idx = np.argmax(business_scores)
    threshold = thresholds[best_idx]

    preds = (probs >= threshold).astype(int)

    precision_val = precision_score(y_test, preds)
    recall_val = recall_score(y_test, preds)
    f1_val = f1_score(y_test, preds)
    accuracy_val = accuracy_score(y_test, preds)

    business_score = (0.7 * recall_val) + (0.3 * precision_val)

    results.append({
        "Model": name,
        "ROC-AUC": roc,
        "PR-AUC": pr_auc,
        "Precision": precision_val,
        "Recall": recall_val,
        "F1": f1_val,
        "Accuracy": accuracy_val,
        "Business Score": business_score,
        "Threshold": threshold
    })

    print(
        f"ROC: {roc:.3f} | PR-AUC: {pr_auc:.3f} | "
        f"Recall: {recall_val:.3f} | Precision: {precision_val:.3f} | "
        f"Score: {business_score:.3f}"
    )

    # Select best
    if business_score > best_business_score:
        best_business_score = business_score
        best_model_name = name

        best_bundle = {
    "model": calibrated,
    "threshold": threshold,
    "features": X.columns.tolist(),
    "categorical_cols": categorical_cols,
    "numerical_cols": numerical_cols,
    "model_name": name
}


# ============================================================
# 📊 9. RESULTS
# ============================================================
results_df = pd.DataFrame(results).sort_values(
    by="Business Score", ascending=False
)

print("\n📊 FINAL MODEL COMPARISON:")
print(results_df)


# ============================================================
# 💾 10. SAVE MODEL
# ============================================================
joblib.dump(best_bundle, "best_fraud_model_v2.pkl")

print(f"\n🏆 Best Model: {best_model_name}")
print("✅ Saved as best_fraud_model_v2.pkl")

Categorical: ['country', 'payment_channel', 'transaction_status', 'currency', 'signup_date', 'acquisition_channel', 'industry']
Numerical: 26 features

🚀 Training Logistic Regression...
ROC: 0.880 | PR-AUC: 0.477 | Recall: 0.996 | Precision: 0.149 | Score: 0.742

🚀 Training Random Forest...
ROC: 0.877 | PR-AUC: 0.431 | Recall: 0.993 | Precision: 0.164 | Score: 0.744

🚀 Training Gradient Boosting...
ROC: 0.897 | PR-AUC: 0.500 | Recall: 0.992 | Precision: 0.190 | Score: 0.751

🚀 Training XGBoost...
ROC: 0.897 | PR-AUC: 0.498 | Recall: 0.993 | Precision: 0.196 | Score: 0.754

📊 FINAL MODEL COMPARISON:
                 Model   ROC-AUC    PR-AUC  Precision    Recall        F1  \
3              XGBoost  0.896773  0.497712   0.196179  0.992530  0.327606   
2    Gradient Boosting  0.897369  0.500208   0.189981  0.992073  0.318895   
1        Random Forest  0.876941  0.430517   0.164205  0.992988  0.281808   
0  Logistic Regression  0.880082  0.477411   0.148816  0.995732  0.258934   

   Accur

uvicorn app.api:app --reload

http://127.0.0.1:8000/

http://127.0.0.1:8000/docs

streamlit run app/sentinel.py

create alert system... tell me about it